In [1]:
from pycelonis import get_celonis

import requests
import pandas as pd

In [2]:
# These are CPM credentials that were created inside the application
url = 'https://envalior.symbioweb.com/envalior/Processworld/_api/rest/facets/processes/views/list/elements'
 
headers = {
    "symbio-auth-token": "REDACTED_SYMBIO_TOKEN",
    "api-version": "2.0",
    "Accept": "application/json"
}
 
TOKEN = "REDACTED_SYMBIO_TOKEN"
 
# Now we extract the name of all processes
response = requests.get(url, headers=headers)
response.raise_for_status()
 
views = response.json()

In [5]:
'''
users = {}
for element in views['elements']:
    authors = element.get('related').get('author')
    for author_aux in authors:
        users[author_aux.get('id')] = author_aux.get('expandUri')
    if element.get('related').get('owner') is not None:
        for owner in element.get('related').get('owner'):
            users[owner.get('id')] = owner.get('expandUri')
'''         
            
users = {}
for element in views['elements']:
    authors = element.get('related').get('author')
    if authors is not None:
        for author_aux in authors:
            users[author_aux.get('id')] = author_aux.get('expandUri')
    owners = element.get('related').get('owner')
    if owners is not None:
        for owner in owners:
            users[owner.get('id')] = owner.get('expandUri')
    additional_authors = element.get('related').get('secondaryAuthors')
    if additional_authors is not None:
        for additional_author in additional_authors:
            users[additional_author.get('id')] = additional_author.get('expandUri')

In [6]:
final_user = []
for key, value in users.items():
    response_u = requests.get(value, headers=headers)
    response_u.raise_for_status()
    views_u = response_u.json()
 
    for aux_user in views_u['elements']:
        if aux_user['type'] == "authToken":
            continue
 
        final_user.append(
            {
                'id': aux_user['id'],
                'name': aux_user['attributes']['firstName']['127'],
                'last name': aux_user['attributes']['lastName']['127'],
                'email': aux_user['attributes']['profileEmail']['127']
 
            }
        )

In [7]:
userDataframe = pd.DataFrame(final_user)

In [8]:
celonis_team_url = 'https://envalior.eu-1.celonis.cloud/'
celonis_api_token = 'REDACTED_CELONIS_TOKEN'

celonis = get_celonis(base_url=celonis_team_url,
                      api_token=celonis_api_token, key_type = 'APP_KEY')

data_pool_id = "6c0151db-f6d2-4d77-b29d-a4f0086b0d8a"
data_pool = celonis.data_integration.get_data_pool(data_pool_id)

[2026-06-11 14:06:05,729] WARNING: Your PyCelonis Version 2.13.0 is outdated (Newest Version: 2.14.2). Please upgrade the package via: pip install --extra-index-url=https://pypi.celonis.cloud/ pycelonis pycelonis_core --upgrade


[2026-06-11 14:06:05,895] INFO: Initial connect successful! PyCelonis Version: 2.13.0
[2026-06-11 14:06:08,637] INFO: `ml-workbench` permissions: []
[2026-06-11 14:06:08,637] INFO: `team` permissions: []
[2026-06-11 14:06:08,638] INFO: `process-repository` permissions: []
[2026-06-11 14:06:08,638] INFO: `storage-manager` permissions: []
[2026-06-11 14:06:08,639] INFO: `package-manager` permissions: []
[2026-06-11 14:06:08,639] INFO: `transformation-center` permissions: []
[2026-06-11 14:06:08,640] INFO: `compute-live` permissions: []
[2026-06-11 14:06:08,641] INFO: `task-mining` permissions: []
[2026-06-11 14:06:08,641] INFO: `event-collection` permissions: []
[2026-06-11 14:06:08,641] INFO: `transformation-hub` permissions: []
[2026-06-11 14:06:08,642] INFO: `user-provisioning` permissions: []
[2026-06-11 14:06:08,642] INFO: `workflows` permissions: []


In [9]:
# Push to celonis
data_pool.create_table(df=userDataframe, table_name="CPMUsers", drop_if_exists=True,force=True)

[2026-06-11 14:06:15,021] WARNING: STRING columns are by default stored as VARCHAR(80) and therefore cut after 80 characters. You can specify a custom field length for each column using the `column_config` parameter.


[2026-06-11 14:06:15,495] INFO: Successfully created data push job with id 'd8e181fe-7f41-4378-a197-d4ef5b26810a'
[2026-06-11 14:06:15,496] INFO: Add data frame as file chunks to data push job with id 'd8e181fe-7f41-4378-a197-d4ef5b26810a'


  0%|          | 0/1 [00:00<?, ?it/s]

[2026-06-11 14:06:15,870] INFO: Successfully upserted file chunk to data push job with id 'd8e181fe-7f41-4378-a197-d4ef5b26810a'
[2026-06-11 14:06:16,929] INFO: Successfully triggered execution for data push job with id 'd8e181fe-7f41-4378-a197-d4ef5b26810a'
[2026-06-11 14:06:16,930] INFO: Wait for execution of data push job with id 'd8e181fe-7f41-4378-a197-d4ef5b26810a'


0it [00:00, ?it/s]

[2026-06-11 14:06:21,380] INFO: Successfully created table 'CPMUsers' in data pool
[2026-06-11 14:06:21,537] INFO: Successfully deleted data push job with id 'd8e181fe-7f41-4378-a197-d4ef5b26810a'


DataPoolTable(name='CPMUsers', data_source_id=None, columns=[], schema_name='6c0151db-f6d2-4d77-b29d-a4f0086b0d8a', data_pool_id='6c0151db-f6d2-4d77-b29d-a4f0086b0d8a')

# Process folders (Dropdown options)

Servicio para obtener carpetas de procesos (`ProcessGroup`) desde la API v2 de Symbio.

In [7]:
from symbio_process_folders import get_process_folder_options, SymbioProcessFoldersError

STORAGE_COLLECTION = "envalior"
TENANT = "Processworld"
BASE_URL = "https://envalior.symbioweb.com"
TOKEN = "REDACTED_SYMBIO_TOKEN"

try:
    # recursive=True expande /v2/data/elements/{id} para obtener todas las carpetas del árbol
    process_folders = get_process_folder_options(
        storagecollection=STORAGE_COLLECTION,
        tenant=TENANT,
        symbio_auth_token=TOKEN,
        base_url=BASE_URL,
        recursive=True,
        folder_types={"subCategory", "processGroup", "category", "mainProcess"},
    )
except SymbioProcessFoldersError as error:
    print(f"Error al obtener carpetas: {error}")
    process_folders = []

print(f"Total carpetas encontradas: {len(process_folders)}")
process_folders[:10]

TypeError: get_process_folder_options() got an unexpected keyword argument 'recursive'

In [8]:
import pandas as pd

process_folders_df = pd.DataFrame(process_folders)
process_folders_df.sort_values("name", na_position="last")

,id,name
0,ab2835f8-da30-4e36-a863-456e1105bb58,Process House
1,883669b6-83af-4ce9-b532-b0ba929bef71,View
